# Ch09 — 變異數分析 (ANOVA)

| 項目 | 說明 |
|------|------|
| **預估時間** | 2 小時 |
| **前置章節** | Ch08 t 檢定與 z 檢定 |
| **資料集** | `datasets/titanic_train.csv`、`datasets/ecommerce_orders.csv` |

## 學習目標

完成本章後，你將能夠：

1. 解釋為何多組比較不能使用多次 t 檢定
2. 理解 ANOVA 的原理：組間變異 vs 組內變異
3. 執行單因子 ANOVA (One-Way ANOVA) 並解讀 F 統計量
4. 計算效果量 (eta-squared) 並判斷實務意義
5. 使用事後檢定 (Post-hoc Test) 找出具體差異配對
6. 在前提假設不滿足時選擇替代方法

## 環境設定

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns

# 中文顯示設定
plt.rcParams["font.family"] = ["Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# 固定隨機種子
np.random.seed(42)

print("環境載入完成")

## 資料載入

In [ ]:
# Titanic 資料集
titanic = pd.read_csv("datasets/titanic_train.csv")
print(f"Titanic 資料集：{titanic.shape[0]} 筆 x {titanic.shape[1]} 欄")
titanic[["Pclass", "Fare", "Age"]].describe()

In [ ]:
# 電商訂單資料集
ecommerce = pd.read_csv("datasets/ecommerce_orders.csv")
print(f"電商資料集：{ecommerce.shape[0]} 筆 x {ecommerce.shape[1]} 欄")
ecommerce.head()

---

## Part A — 為什麼需要 ANOVA？

### 多組比較問題

Titanic 有三個艙等 (Pclass = 1, 2, 3)，我們想知道：**不同艙等的票價 (Fare) 是否有顯著差異？**

直覺做法：對每一對艙等進行獨立的 t 檢定 (t-test)。但這樣會有嚴重的統計問題。

In [ ]:
# 先看看三個艙等的票價分布
fare_by_class = titanic.groupby("Pclass")["Fare"].describe()
fare_by_class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 箱形圖
sns.boxplot(data=titanic, x="Pclass", y="Fare", ax=axes[0])
axes[0].set_title("各艙等票價分布")
axes[0].set_xlabel("艙等 (Pclass)")
axes[0].set_ylabel("票價 (Fare)")

# 直方圖
for pclass in [1, 2, 3]:
    subset = titanic[titanic["Pclass"] == pclass]["Fare"]
    axes[1].hist(subset, bins=30, alpha=0.5, label=f"Pclass {pclass}")
axes[1].set_title("各艙等票價直方圖")
axes[1].set_xlabel("票價 (Fare)")
axes[1].set_ylabel("人數")
axes[1].legend()

plt.tight_layout()
plt.show()

### 多重 t-test 的問題：alpha 膨脹 (Familywise Error Rate)

如果我們對 $k$ 組做所有成對 t 檢定，需要進行 $\binom{k}{2}$ 次比較。

每次檢定的 Type I Error 為 $\alpha$，但**整體犯錯機率 (familywise error rate)** 為：

$$\alpha_{\text{family}} = 1 - (1 - \alpha)^{C}$$

其中 $C = \binom{k}{2}$ 是比較次數。

In [ ]:
from math import comb

alpha = 0.05

print("多重比較的 alpha 膨脹：")
print(f"{'組數 k':>6} | {'比較次數 C':>10} | {'整體犯錯率':>12}")
print("-" * 38)

for k in [2, 3, 5, 10, 20]:
    c = comb(k, 2)
    familywise_alpha = 1 - (1 - alpha) ** c
    print(f"{k:>6} | {c:>10} | {familywise_alpha:>11.1%}")

以 Titanic 的 3 個艙等為例：
- 需要 $\binom{3}{2} = 3$ 次成對比較
- 整體犯錯率 = $1 - (1 - 0.05)^3 \approx 14.3\%$
- 即使所有組別實際上沒有差異，仍有 **14.3%** 的機率誤判至少一對有差異！

ANOVA 用**一次檢定**同時比較所有組別，避免 alpha 膨脹。

### ANOVA 的邏輯：組間變異 vs 組內變異

ANOVA 的核心概念是將**總變異**拆解為兩部分：

| 變異來源 | 說明 | 縮寫 |
|----------|------|------|
| **組間變異** (Between-group) | 各組平均數之間的差異 | SSB |
| **組內變異** (Within-group) | 各觀測值與其所屬組別平均數的差異 | SSW |
| **總變異** (Total) | 所有觀測值與總平均數的差異 | SST |

$$SS_{\text{total}} = SS_{\text{between}} + SS_{\text{within}}$$

如果組間變異相對於組內變異很大，代表各組平均數之間的差異可能不只是隨機波動。

In [ ]:
# 視覺化：組間 vs 組內變異
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 情境 1：組間變異大 >> 組內變異 → 顯著差異
np.random.seed(42)
g1 = np.random.normal(loc=10, scale=1, size=50)
g2 = np.random.normal(loc=15, scale=1, size=50)
g3 = np.random.normal(loc=20, scale=1, size=50)

for i, (g, label) in enumerate([(g1, "A"), (g2, "B"), (g3, "C")]):
    axes[0].scatter([i+1]*50, g, alpha=0.5, label=label)
    axes[0].hlines(g.mean(), i+0.7, i+1.3, color="red", linewidth=2)

axes[0].set_title("組間變異大 → ANOVA 顯著")
axes[0].set_xticks([1, 2, 3])
axes[0].set_xticklabels(["A", "B", "C"])

# 情境 2：組間變異小 ≈ 組內變異 → 不顯著
g1 = np.random.normal(loc=15, scale=3, size=50)
g2 = np.random.normal(loc=15.5, scale=3, size=50)
g3 = np.random.normal(loc=16, scale=3, size=50)

for i, (g, label) in enumerate([(g1, "A"), (g2, "B"), (g3, "C")]):
    axes[1].scatter([i+1]*50, g, alpha=0.5, label=label)
    axes[1].hlines(g.mean(), i+0.7, i+1.3, color="red", linewidth=2)

axes[1].set_title("組間變異小 → ANOVA 不顯著")
axes[1].set_xticks([1, 2, 3])
axes[1].set_xticklabels(["A", "B", "C"])

plt.tight_layout()
plt.show()

---

## Part B — 單因子 ANOVA (One-Way ANOVA)

### F 統計量

ANOVA 使用 **F 統計量** 來衡量組間差異的顯著性：

$$F = \frac{MSB}{MSW} = \frac{SS_{\text{between}} / (k - 1)}{SS_{\text{within}} / (N - k)}$$

其中：
- $MSB$ = 組間均方 (Mean Square Between)
- $MSW$ = 組內均方 (Mean Square Within)
- $k$ = 組數
- $N$ = 總觀測數

**F 值越大，表示組間差異相對於組內變異越顯著。**

### 前提假設

ANOVA 需要滿足三個前提假設：

1. **常態性 (Normality)**：各組資料近似常態分布
2. **方差齊性 (Homogeneity of Variance)**：各組的方差相等
3. **獨立性 (Independence)**：觀測值之間相互獨立

### 假設檢驗

- $H_0$：所有艙等的平均票價相同 ($\mu_1 = \mu_2 = \mu_3$)
- $H_1$：至少有一對艙等的平均票價不同

In [ ]:
# 前提檢查 1：常態性 (Shapiro-Wilk test)
print("=== Shapiro-Wilk 常態性檢定 ===")
print(f"H0: 資料服從常態分布\n")

for pclass in [1, 2, 3]:
    fare_data = titanic[titanic["Pclass"] == pclass]["Fare"].dropna()
    stat, p = stats.shapiro(fare_data)
    result = "拒絕 H0 (非常態)" if p < 0.05 else "無法拒絕 H0 (近似常態)"
    print(f"Pclass {pclass}: W = {stat:.4f}, p = {p:.4e} → {result}")

In [ ]:
# 前提檢查 2：方差齊性 (Levene's test)
class1_fare = titanic[titanic["Pclass"] == 1]["Fare"].dropna()
class2_fare = titanic[titanic["Pclass"] == 2]["Fare"].dropna()
class3_fare = titanic[titanic["Pclass"] == 3]["Fare"].dropna()

stat, p = stats.levene(class1_fare, class2_fare, class3_fare)
print("=== Levene 方差齊性檢定 ===")
print(f"H0: 各組方差相等")
print(f"Levene stat = {stat:.4f}, p = {p:.4e}")
print(f"結論：{'拒絕 H0 (方差不齊)' if p < 0.05 else '無法拒絕 H0 (方差齊性成立)'}")

> **注意**：Titanic Fare 的常態性和方差齊性可能不滿足（票價分布右偏、高等艙方差較大）。
> 但 ANOVA 對常態性偏離有一定的穩健性 (robustness)，特別是當各組樣本數較大時 (中央極限定理)。
> 若嚴重違反前提，我們會在 Part E 介紹替代方法。
> 這裡我們先進行 ANOVA 作為教學示範。

In [ ]:
# 執行單因子 ANOVA
f_stat, p_value = stats.f_oneway(class1_fare, class2_fare, class3_fare)

print("=== One-Way ANOVA 結果 ===")
print(f"F 統計量 = {f_stat:.4f}")
print(f"p-value  = {p_value:.4e}")
print()

alpha = 0.05
if p_value < alpha:
    print(f"p = {p_value:.4e} < alpha = {alpha}")
    print("結論：拒絕 H0 — 至少有一對艙等的平均票價存在顯著差異")
else:
    print(f"p = {p_value:.4e} >= alpha = {alpha}")
    print("結論：無法拒絕 H0 — 無足夠證據認為艙等間票價有差異")

### F 值解讀

- **F 值越大**：組間差異相對於組內變異越顯著
- **F = 1**：組間差異 = 組內變異（沒有差異的期望值）
- **F >> 1**：強烈暗示各組平均數不同
- **p-value**：在 H0 為真的情況下，觀察到如此極端或更極端 F 值的機率

---

## Part C — 效果量：eta-squared ($\eta^2$)

### 為什麼需要效果量？

p-value 只告訴你差異是否**統計顯著**，但不告訴你差異有多**大**。

當樣本數非常大時，即使很微小的差異也會達到統計顯著。因此我們需要**效果量 (Effect Size)** 來衡量實務意義。

### eta-squared 公式

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

| $\eta^2$ 範圍 | 效果大小 |
|:-:|:-:|
| 0.01 | 小效果 (Small) |
| 0.06 | 中效果 (Medium) |
| 0.14 | 大效果 (Large) |

In [ ]:
# 手動計算 eta-squared
fare_all = titanic["Fare"].dropna()
grand_mean = fare_all.mean()

# SS_between：各組平均與總平均的差異
ss_between = sum(
    len(titanic[titanic["Pclass"] == pc]["Fare"].dropna())
    * (titanic[titanic["Pclass"] == pc]["Fare"].dropna().mean() - grand_mean) ** 2
    for pc in [1, 2, 3]
)

# SS_total：所有觀測值與總平均的差異
ss_total = ((fare_all - grand_mean) ** 2).sum()

# SS_within = SS_total - SS_between
ss_within = ss_total - ss_between

eta_squared = ss_between / ss_total

print("=== 變異數分解 ===")
print(f"SS_between = {ss_between:,.2f}")
print(f"SS_within  = {ss_within:,.2f}")
print(f"SS_total   = {ss_total:,.2f}")
print()
print(f"eta-squared = {eta_squared:.4f}")

if eta_squared >= 0.14:
    print(f"解讀：大效果 (Large) — 艙等解釋了票價 {eta_squared:.1%} 的變異")
elif eta_squared >= 0.06:
    print(f"解讀：中效果 (Medium) — 艙等解釋了票價 {eta_squared:.1%} 的變異")
else:
    print(f"解讀：小效果 (Small) — 艙等解釋了票價 {eta_squared:.1%} 的變異")

---

## Part D — 事後檢定 (Post-hoc Tests)

### 為什麼需要事後檢定？

ANOVA 只告訴你**「有差」**，但不告訴你**「哪裡差」**。

- ANOVA 顯著 → 至少有一對組別平均數不同
- 但是哪一對？Pclass 1 vs 2？1 vs 3？2 vs 3？

事後檢定 (Post-hoc Test) 用於在 ANOVA 顯著後，找出具體是哪些配對有差異。

### Tukey HSD (Honestly Significant Difference)

Tukey HSD 是最常用的事後檢定方法，它同時控制所有成對比較的整體 Type I Error Rate。

In [ ]:
# 準備資料：需要移除 Fare 為 NaN 的列
titanic_clean = titanic[["Pclass", "Fare"]].dropna()

# 執行 Tukey HSD
tukey_result = pairwise_tukeyhsd(
    endog=titanic_clean["Fare"],       # 數值變數
    groups=titanic_clean["Pclass"],    # 分組變數
    alpha=0.05                          # 顯著水準
)

print("=== Tukey HSD 事後檢定結果 ===")
print(tukey_result.summary())

In [ ]:
# Tukey HSD 視覺化
fig, ax = plt.subplots(figsize=(10, 5))
tukey_result.plot_simultaneous(ax=ax)
ax.set_title("Tukey HSD 95% 同時信賴區間")
ax.set_xlabel("票價 (Fare)")
ax.set_ylabel("艙等 (Pclass)")
plt.tight_layout()
plt.show()

### 解讀 Tukey HSD 結果

- **meandiff**：兩組平均數的差異
- **lower / upper**：差異的 95% 信賴區間
- **reject**：是否拒絕 H0（True = 有顯著差異）

在視覺化圖中，信賴區間不重疊的組別之間存在顯著差異。

### Bonferroni 校正

另一種常見的多重比較校正方法是 **Bonferroni 校正**：

$$\alpha_{\text{Bonferroni}} = \frac{\alpha}{C}$$

其中 $C$ 為比較次數。

Bonferroni 比 Tukey HSD 更保守（更不容易拒絕 H0），但適用範圍更廣。

In [ ]:
# Bonferroni 校正示範
from itertools import combinations

pclass_labels = [1, 2, 3]
pairs = list(combinations(pclass_labels, 2))
n_comparisons = len(pairs)
bonferroni_alpha = alpha / n_comparisons

print(f"原始 alpha = {alpha}")
print(f"比較次數 = {n_comparisons}")
print(f"Bonferroni 校正後 alpha = {bonferroni_alpha:.4f}")
print()

print(f"{'配對':>12} | {'t-stat':>8} | {'p-value':>10} | {'顯著？':>8}")
print("-" * 50)

for pc1, pc2 in pairs:
    g1 = titanic[titanic["Pclass"] == pc1]["Fare"].dropna()
    g2 = titanic[titanic["Pclass"] == pc2]["Fare"].dropna()
    t_stat, p_val = stats.ttest_ind(g1, g2, equal_var=False)
    sig = "Yes" if p_val < bonferroni_alpha else "No"
    print(f"  {pc1} vs {pc2}    | {t_stat:>8.3f} | {p_val:>10.4e} | {sig:>8}")

---

## Part E — 前提不滿足怎麼辦？

### 方差齊性不滿足 → Welch's ANOVA

當各組方差不相等時（Levene 檢定拒絕 H0），可以使用 **Welch's ANOVA**，
它不假設方差齊性。

在 SciPy 中，我們可以使用 `scipy.stats.alexandergovern()` (Alexander-Govern test)
或使用 `pingouin` 套件的 `welch_anova()`。這裡我們用 SciPy 內建方法：

In [ ]:
# Welch's ANOVA（不假設方差齊性）
# scipy.stats.alexandergovern 等效於 Welch's ANOVA
result = stats.alexandergovern(class1_fare, class2_fare, class3_fare)

print("=== Welch's ANOVA (Alexander-Govern Test) ===")
print(f"統計量 = {result.statistic:.4f}")
print(f"p-value = {result.pvalue:.4e}")
print(f"結論：{'拒絕 H0 — 有顯著差異' if result.pvalue < 0.05 else '無法拒絕 H0'}")

### 常態性不滿足 → Kruskal-Wallis 檢定

當資料嚴重偏離常態分布時，可以使用**無母數替代方法**：

- **Kruskal-Wallis H 檢定**：ANOVA 的無母數版本，比較各組的中位數排名
- 詳細內容將在 **Ch10 無母數檢定** 中介紹

這裡先做一個預覽：

In [ ]:
# Kruskal-Wallis 檢定（預覽）
h_stat, p_value_kw = stats.kruskal(class1_fare, class2_fare, class3_fare)

print("=== Kruskal-Wallis H 檢定（ANOVA 的無母數替代）===")
print(f"H 統計量 = {h_stat:.4f}")
print(f"p-value  = {p_value_kw:.4e}")
print(f"結論：{'拒絕 H0 — 有顯著差異' if p_value_kw < 0.05 else '無法拒絕 H0'}")
print()
print("更多無母數檢定方法，請見 Ch10。")

### 知識補給站：雙因子 ANOVA (Two-Way ANOVA) 簡介

單因子 ANOVA 只考慮**一個**分類自變數對依變數的影響。

**雙因子 ANOVA** 可以同時分析**兩個**分類自變數，並檢測：

1. **主效果 A** (Main Effect A)：因子 A 是否影響依變數
2. **主效果 B** (Main Effect B)：因子 B 是否影響依變數
3. **交互作用效果** (Interaction Effect)：因子 A 和 B 的組合是否產生額外影響

例如：Titanic 中，**艙等 (Pclass)** 和 **性別 (Sex)** 是否對票價有交互作用？

```python
# 雙因子 ANOVA 語法（使用 statsmodels）
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols('Fare ~ C(Pclass) * C(Sex)', data=titanic).fit()
sm.stats.anova_lm(model, typ=2)
```

雙因子 ANOVA 超出本章範圍，但概念上是單因子 ANOVA 的自然延伸。

### 方法選擇決策樹

```
多組比較
 |
 +-- 常態性成立？
      |
      +-- Yes → 方差齊性成立？
      |         |
      |         +-- Yes → One-Way ANOVA
      |         |         事後：Tukey HSD
      |         |
      |         +-- No  → Welch's ANOVA
      |                   事後：Games-Howell
      |
      +-- No  → Kruskal-Wallis
                事後：Dunn's test
```

---

## Part F — 實務範例：電商品類價格差異

### 問題

電商平台有多個商品品類 (category)，我們想知道：**不同品類的商品單價 (unit_price) 是否有顯著差異？**

我們將遵循完整的分析流程：

1. 探索性分析 (EDA)
2. 前提檢查（Shapiro → Levene）
3. 選擇適當的檢定方法
4. 執行檢定
5. 事後檢定

In [ ]:
# Step 1：探索性分析
print("=== 各品類統計摘要 ===")
category_stats = ecommerce.groupby("category")["unit_price"].describe()
print(category_stats)
print()
print(f"品類數量：{ecommerce['category'].nunique()}")
print(f"各品類：{ecommerce['category'].unique().tolist()}")

In [ ]:
# 視覺化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 箱形圖
sns.boxplot(data=ecommerce, x="category", y="unit_price", ax=axes[0])
axes[0].set_title("各品類商品單價分布")
axes[0].set_xlabel("品類")
axes[0].set_ylabel("單價 (unit_price)")
axes[0].tick_params(axis="x", rotation=30)

# 小提琴圖
sns.violinplot(data=ecommerce, x="category", y="unit_price", ax=axes[1])
axes[1].set_title("各品類商品單價小提琴圖")
axes[1].set_xlabel("品類")
axes[1].set_ylabel("單價 (unit_price)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Step 2：前提檢查 — 常態性
categories = ecommerce["category"].unique()

print("=== Shapiro-Wilk 常態性檢定 ===")
normality_ok = True
for cat in sorted(categories):
    data = ecommerce[ecommerce["category"] == cat]["unit_price"].dropna()
    if len(data) >= 3:  # Shapiro 需要至少 3 筆資料
        stat, p = stats.shapiro(data)
        result = "常態" if p >= 0.05 else "非常態"
        if p < 0.05:
            normality_ok = False
        print(f"{cat:>15}: W = {stat:.4f}, p = {p:.4f} → {result}")
    else:
        print(f"{cat:>15}: 樣本數不足 (n={len(data)})")

print(f"\n整體常態性：{'通過' if normality_ok else '部分品類不滿足'}")

In [ ]:
# Step 2：前提檢查 — 方差齊性
groups_data = [ecommerce[ecommerce["category"] == cat]["unit_price"].dropna()
               for cat in sorted(categories)]

levene_stat, levene_p = stats.levene(*groups_data)

print("=== Levene 方差齊性檢定 ===")
print(f"統計量 = {levene_stat:.4f}")
print(f"p-value = {levene_p:.4f}")
variance_ok = levene_p >= 0.05
print(f"結論：{'方差齊性成立' if variance_ok else '方差不齊'}")

In [ ]:
# Step 3 & 4：根據前提檢查結果選擇適當方法
print("=== 檢定方法選擇 ===")
print(f"常態性：{'通過' if normality_ok else '不通過'}")
print(f"方差齊性：{'通過' if variance_ok else '不通過'}")

if normality_ok and variance_ok:
    print("→ 使用 One-Way ANOVA")
    f_stat_ec, p_value_ec = stats.f_oneway(*groups_data)
    test_name = "One-Way ANOVA"
elif normality_ok and not variance_ok:
    print("→ 使用 Welch's ANOVA (Alexander-Govern)")
    result_ec = stats.alexandergovern(*groups_data)
    f_stat_ec, p_value_ec = result_ec.statistic, result_ec.pvalue
    test_name = "Welch's ANOVA"
else:
    print("→ 使用 Kruskal-Wallis 檢定（無母數方法）")
    f_stat_ec, p_value_ec = stats.kruskal(*groups_data)
    test_name = "Kruskal-Wallis"

print(f"\n=== {test_name} 結果 ===")
print(f"統計量 = {f_stat_ec:.4f}")
print(f"p-value = {p_value_ec:.4e}")
print(f"結論：{'拒絕 H0 — 品類間單價有顯著差異' if p_value_ec < 0.05 else '無法拒絕 H0 — 無足夠證據'}")

In [ ]:
# Step 5：計算效果量
price_all = ecommerce["unit_price"].dropna()
grand_mean_ec = price_all.mean()

ss_between_ec = sum(
    len(ecommerce[ecommerce["category"] == cat]["unit_price"].dropna())
    * (ecommerce[ecommerce["category"] == cat]["unit_price"].dropna().mean() - grand_mean_ec) ** 2
    for cat in categories
)
ss_total_ec = ((price_all - grand_mean_ec) ** 2).sum()
eta_sq_ec = ss_between_ec / ss_total_ec

print(f"=== 效果量 ===")
print(f"eta-squared = {eta_sq_ec:.4f}")

if eta_sq_ec >= 0.14:
    print(f"解讀：大效果 — 品類解釋了單價 {eta_sq_ec:.1%} 的變異")
elif eta_sq_ec >= 0.06:
    print(f"解讀：中效果 — 品類解釋了單價 {eta_sq_ec:.1%} 的變異")
else:
    print(f"解讀：小效果 — 品類解釋了單價 {eta_sq_ec:.1%} 的變異")

In [ ]:
# Step 6：事後檢定
ecommerce_clean = ecommerce[["category", "unit_price"]].dropna()

tukey_ec = pairwise_tukeyhsd(
    endog=ecommerce_clean["unit_price"],
    groups=ecommerce_clean["category"],
    alpha=0.05
)

print("=== Tukey HSD 事後檢定（電商品類）===")
print(tukey_ec.summary())

In [ ]:
# 視覺化 Tukey 結果
fig, ax = plt.subplots(figsize=(10, 6))
tukey_ec.plot_simultaneous(ax=ax)
ax.set_title("Tukey HSD 95% 同時信賴區間（電商品類）")
ax.set_xlabel("商品單價 (unit_price)")
ax.set_ylabel("品類 (category)")
plt.tight_layout()
plt.show()

In [ ]:
# 完整結果彙整
print("=" * 55)
print("    電商品類單價差異分析 — 完整報告")
print("=" * 55)
print(f"\n1. 常態性：{'通過' if normality_ok else '部分品類不通過'}")
print(f"2. 方差齊性：{'通過' if variance_ok else '不通過 (Levene p={levene_p:.4f})'}")
print(f"3. 使用方法：{test_name}")
print(f"4. 統計量 = {f_stat_ec:.4f}, p = {p_value_ec:.4e}")
print(f"5. 效果量 eta-squared = {eta_sq_ec:.4f}")
print(f"6. 結論：{'品類間單價有顯著差異' if p_value_ec < 0.05 else '無足夠證據認為品類間單價有差異'}")

if p_value_ec < 0.05:
    # 列出有顯著差異的配對
    sig_pairs = []
    for i in range(len(tukey_ec.reject)):
        if tukey_ec.reject[i]:
            g1 = tukey_ec.groupsunique[tukey_ec._results_table.data[i+1][0]]\
                if hasattr(tukey_ec, '_results_table') else ""
    print(f"\n7. 事後檢定顯著配對：見上方 Tukey HSD 表格 (reject = True)")

---

## 重點整理

| 概念 | 說明 |
|------|------|
| **ANOVA** | 比較 3 組以上平均數是否相同的檢定方法 |
| **F 統計量** | MSB / MSW，衡量組間 vs 組內變異的比值 |
| **alpha 膨脹** | 多重 t-test 導致整體犯錯率上升 |
| **eta-squared** | 效果量指標，SS_between / SS_total |
| **Tukey HSD** | 最常用的事後檢定，找出具體差異配對 |
| **Bonferroni** | 保守的多重比較校正方法 |
| **Welch's ANOVA** | 方差不齊時的替代方法 |
| **Kruskal-Wallis** | 常態性不滿足時的無母數替代方法 |

## 練習題

### 基礎練習

1. 使用 Titanic 資料集，檢定不同**登船港口 (Embarked)** 的乘客年齡 (Age) 是否有顯著差異。
   - 進行前提檢查（常態性 + 方差齊性）
   - 執行適當的 ANOVA 或替代方法
   - 計算 eta-squared

### 進階練習

2. 使用電商資料集，分析不同品類的**訂單金額 (amount)** 是否有顯著差異。
   - 完成完整流程：EDA → 前提檢查 → 檢定 → 效果量 → 事後檢定
   - 將結果視覺化並撰寫結論

### 挑戰練習

3. 使用 Titanic 資料集進行**雙因子 ANOVA**：
   - 因子 A：艙等 (Pclass)
   - 因子 B：性別 (Sex)
   - 依變數：票價 (Fare)
   - 使用 `statsmodels` 的 `ols` + `anova_lm` 方法
   - 解讀主效果和交互作用效果

---

← [Ch08 t 檢定與 z 檢定](08_t檢定與z檢定.ipynb) | [Ch10 無母數檢定 →](10_無母數檢定.ipynb)